# 01 · Data overview and cleaning

This notebook checks the restaurant data, documents the cleaning decisions and builds one table with **one row per restaurant per day**: covers served, calendar information, restaurant type and the reservations that were already known the day before.

**Dataset:** Recruit Restaurant Visitor Forecasting (Kaggle, 2017). Daily covers of ~800 restaurants in Japan from January 2016 to April 2017, recorded by a point-of-sale system, plus online reservations from two booking platforms.

| File | Content |
|---|---|
| `air_visit_data.csv` | Covers per restaurant per day (days with no row = closed) |
| `air_reserve.csv` | Reservations made through booking platform A |
| `hpg_reserve.csv` | Reservations made through booking platform B |
| `store_id_relation.csv` | Restaurants present on both platforms |
| `air_store_info.csv` | Cuisine and area of each restaurant |
| `date_info.csv` | Calendar with day of week and public holidays |

In [1]:
import pandas as pd
import numpy as np

RAW = "../data/raw/"

visits = pd.read_csv(RAW + "air_visit_data.csv", parse_dates=["visit_date"])
air_res = pd.read_csv(RAW + "air_reserve.csv", parse_dates=["visit_datetime", "reserve_datetime"])
hpg_res = pd.read_csv(RAW + "hpg_reserve.csv", parse_dates=["visit_datetime", "reserve_datetime"])
relation = pd.read_csv(RAW + "store_id_relation.csv")
stores = pd.read_csv(RAW + "air_store_info.csv")
calendar = pd.read_csv(RAW + "date_info.csv", parse_dates=["calendar_date"])

for name, t in {"visits": visits, "air_res": air_res, "hpg_res": hpg_res,
                "relation": relation, "stores": stores, "calendar": calendar}.items():
    print(f"{name:<9} {t.shape[0]:>10,} rows  {t.shape[1]} columns")

visits       252,108 rows  3 columns
air_res       92,378 rows  4 columns
hpg_res    2,000,320 rows  4 columns
relation         150 rows  2 columns
stores           829 rows  5 columns
calendar         517 rows  3 columns


## 1.1 Covers per restaurant per day

In [2]:
print(f"Restaurants: {visits['air_store_id'].nunique()}")
print(f"Period: {visits['visit_date'].min().date()} -> {visits['visit_date'].max().date()}")
print(f"Duplicated restaurant-days: {visits.duplicated(['air_store_id', 'visit_date']).sum()}")
print(f"Missing values: {visits.isna().sum().sum()}")
visits["visitors"].describe()

Restaurants: 829
Period: 2016-01-01 -> 2017-04-22
Duplicated restaurant-days: 0
Missing values: 0


count    252108.000000
mean         20.973761
std          16.757007
min           1.000000
25%           9.000000
50%          17.000000
75%          29.000000
max         877.000000
Name: visitors, dtype: float64

Covers per restaurant-day are strongly right-skewed: most days are small, a few are very large. The extreme values are inspected before deciding what to do with them.

In [3]:
store_median = visits.groupby("air_store_id")["visitors"].transform("median")
visits["ratio_to_median"] = visits["visitors"] / store_median
print(f"Restaurant-days above 10x the restaurant's median: {(visits['ratio_to_median'] > 10).sum()}")
visits.sort_values("visitors", ascending=False).head(5)

Restaurant-days above 10x the restaurant's median: 89


,air_store_id,visit_date,visitors,ratio_to_median
85314,air_cfdeb326418194ff,2017-03-08,877,58.466667
214825,air_8c3175aa5e4fc569,2017-04-18,777,26.793103
72836,air_f2985de32bb792e0,2016-07-10,675,23.275862
172123,air_eca5e0064dc9314a,2016-08-30,627,21.620690
143894,air_43d577e0c9460e64,2016-01-24,514,16.580645


Very high days are kept: in a restaurant they can be real (private events, group bookings, holidays), and removing them would make the forecast look better than it is. They are flagged with `is_extreme` so their effect can be checked.

In [4]:
visits["is_extreme"] = (visits["ratio_to_median"] > 10).astype(int)
visits = visits.drop(columns="ratio_to_median")

## 1.2 When did each restaurant start reporting, and how often is it closed?

In [5]:
span = visits.groupby("air_store_id")["visit_date"].agg(first="min", last="max", open_days="count")
span["days_in_span"] = (span["last"] - span["first"]).dt.days + 1
span["open_share"] = span["open_days"] / span["days_in_span"]

print("Restaurants by first reporting month:")
print(span["first"].dt.to_period("M").value_counts().sort_index().head(8))
print()
print(f"Median share of days open between first and last report: {span['open_share'].median():.1%}")

Restaurants by first reporting month:
first
2016-01    302
2016-02      4
2016-03      6
2016-04      2
2016-05      2
2016-07    485
2016-08      5
2016-09      7
Freq: M, Name: count, dtype: int64

Median share of days open between first and last report: 85.9%


Restaurants do not all start reporting on the same day, so **total covers are not comparable over time**: covers **per open restaurant** are used instead. A day without a row means the restaurant was closed (or did not report): those days are not forecast, and only open days are evaluated.

## 1.3 Reservations: when are they made?

For a restaurant planning tomorrow's prep, only reservations made **before the day of the visit** are known. Reservations made on the day itself would be information from the future: they are kept for description but not used to forecast.

In [6]:
hpg_res = hpg_res.merge(relation, on="hpg_store_id", how="inner")  # keep restaurants present in both systems

for name, r in {"Platform A": air_res, "Platform B": hpg_res}.items():
    lead_days = (r["visit_datetime"].dt.normalize() - r["reserve_datetime"].dt.normalize()).dt.days
    print(f"{name}: {len(r):,} reservations, {r['air_store_id'].nunique()} restaurants, "
          f"same-day share {(lead_days == 0).mean():.1%}, negative lead {(lead_days < 0).sum()}, "
          f"median lead {lead_days.median():.0f} days")

Platform A: 92,378 reservations, 314 restaurants, same-day share 29.6%, negative lead 0, median lead 2 days
Platform B: 28,183 reservations, 150 restaurants, same-day share 11.9%, negative lead 0, median lead 4 days


In [7]:
def daily_reservations(res, prefix):
    """Reserved covers per restaurant per visit day: all, and only those known by the day before."""
    r = res.copy()
    r["visit_date"] = r["visit_datetime"].dt.normalize()
    known = r["reserve_datetime"].dt.normalize() < r["visit_date"]
    all_res = r.groupby(["air_store_id", "visit_date"])["reserve_visitors"].sum().rename(f"{prefix}_all")
    d1_res = r[known].groupby(["air_store_id", "visit_date"])["reserve_visitors"].sum().rename(f"{prefix}_d1")
    return pd.concat([all_res, d1_res], axis=1).fillna(0).reset_index()

res_a = daily_reservations(air_res, "res_a")
res_b = daily_reservations(hpg_res, "res_b")

monthly = (pd.concat([res_a.assign(platform="A"), res_b.assign(platform="B")])
             .assign(month=lambda d: d["visit_date"].dt.to_period("M"),
                     reserved=lambda d: d["res_a_all"].fillna(0) + d["res_b_all"].fillna(0))
             .pivot_table(index="month", columns="platform", values="reserved", aggfunc="sum"))
monthly

platform,A,B
month,,
2016-01,10335.0,4455.0
2016-02,11579.0,5566.0
2016-03,17071.0,7997.0
2016-04,13811.0,6301.0
2016-05,8385.0,5678.0
2016-06,4590.0,6261.0
2016-07,4603.0,6794.0
2016-08,2.0,6214.0
2016-09,12.0,6370.0


The monthly totals show whether the reservation feed is stable over time. Any month with an abrupt drop in reserved covers would point to a data gap rather than a change in behaviour, and should be read with care in the analysis.

## 1.4 Build the daily table

In [8]:
stores["prefecture"] = stores["air_area_name"].str.split().str[0]
calendar = calendar.rename(columns={"calendar_date": "visit_date"})

daily = (visits
         .merge(calendar, on="visit_date", how="left")
         .merge(stores[["air_store_id", "air_genre_name", "prefecture"]], on="air_store_id", how="left")
         .merge(res_a, on=["air_store_id", "visit_date"], how="left")
         .merge(res_b, on=["air_store_id", "visit_date"], how="left"))

res_cols = ["res_a_all", "res_a_d1", "res_b_all", "res_b_d1"]
daily[res_cols] = daily[res_cols].fillna(0)
daily["res_d1"] = daily["res_a_d1"] + daily["res_b_d1"]       # known the day before
daily["res_all"] = daily["res_a_all"] + daily["res_b_all"]    # all, including same-day

with_res = set(res_a["air_store_id"]) | set(res_b["air_store_id"])
daily["has_reservations"] = daily["air_store_id"].isin(with_res).astype(int)

print(daily.shape)
print(f"Missing values: {daily.isna().sum().sum()}")
print(f"Restaurants with reservation data: {daily.loc[daily['has_reservations'] == 1, 'air_store_id'].nunique()} "
      f"of {daily['air_store_id'].nunique()}")
daily.head()

(252108, 15)
Missing values: 0
Restaurants with reservation data: 333 of 829


,air_store_id,visit_date,visitors,is_extreme,day_of_week,holiday_flg,air_genre_name,prefecture,res_a_all,res_a_d1,res_b_all,res_b_d1,res_d1,res_all,has_reservations
0,air_ba937bf13d40fb24,2016-01-13,25,0,Wednesday,0,Dining bar,Tōkyō-to,0.0,0.0,0.0,0.0,0.0,0.0,0
1,air_ba937bf13d40fb24,2016-01-14,32,0,Thursday,0,Dining bar,Tōkyō-to,0.0,0.0,0.0,0.0,0.0,0.0,0
2,air_ba937bf13d40fb24,2016-01-15,29,0,Friday,0,Dining bar,Tōkyō-to,0.0,0.0,0.0,0.0,0.0,0.0,0
3,air_ba937bf13d40fb24,2016-01-16,22,0,Saturday,0,Dining bar,Tōkyō-to,0.0,0.0,0.0,0.0,0.0,0.0,0
4,air_ba937bf13d40fb24,2016-01-18,6,0,Monday,0,Dining bar,Tōkyō-to,0.0,0.0,0.0,0.0,0.0,0.0,0


In [9]:
daily.to_csv("../data/processed/daily_visits.csv", index=False)

## Summary

In [10]:
print(f"Restaurant-days: {len(daily):,} | restaurants: {daily['air_store_id'].nunique()} | "
      f"period: {daily['visit_date'].min().date()} -> {daily['visit_date'].max().date()}")
print(f"Median covers per restaurant-day: {daily['visitors'].median():.0f} (mean {daily['visitors'].mean():.1f})")
print(f"Extreme days (>10x restaurant median): {daily['is_extreme'].sum()}")
print(f"Median share of days open: {span['open_share'].median():.1%}")
print(f"Restaurants with reservation data: {daily.loc[daily['has_reservations'] == 1, 'air_store_id'].nunique()}")
print(f"Reserved covers known the day before: {daily['res_d1'].sum() / daily['res_all'].sum():.1%} of all reserved covers")

Restaurant-days: 252,108 | restaurants: 829 | period: 2016-01-01 -> 2017-04-22
Median covers per restaurant-day: 17 (mean 21.0)
Extreme days (>10x restaurant median): 89
Median share of days open: 85.9%
Restaurants with reservation data: 333
Reserved covers known the day before: 81.1% of all reserved covers


## Cleaning decisions

| # | Topic | Decision |
|---|---|---|
| 1 | Closed days (no row) | Not forecast, not evaluated |
| 2 | Restaurants start reporting at different dates | Compare covers per open restaurant, not totals |
| 3 | Very high days | Kept and flagged with `is_extreme` (events are real demand) |
| 4 | Restaurants on platform B | Linked through `store_id_relation.csv`; others dropped |
| 5 | Same-day reservations | Described, but only reservations made before the visit day (`res_d1`) are used to forecast |

**Key facts:** 252,108 restaurant-days from 829 restaurants (January 2016 – April 2017). A typical restaurant serves 17 covers a day (mean 21.0) and is open on 85.9% of days. 89 days are extreme (more than 10x the restaurant's median). 333 restaurants have online reservation data, and 81.1% of their reserved covers are booked before the day of the visit.